In [1]:
import os
import json

In [2]:
from cimgraph.databases.fileparsers import XMLFile
from cimgraph.databases import ConnectionParameters
from cimgraph.models import FeederModel, DistributedArea
import cimgraph.data_profile.cimhub_2023 as cim 
import cimgraph.utils as utils
from uuid import UUID

In [3]:
params = ConnectionParameters(filename="/home/ande188/Powergrid-Models/models/feeders/CIM/XML/IEEE13/IEEE13.xml", cim_profile='cimhub_2023', iec61970_301=8)
xml_file = XMLFile(params)
feeder_mrid = "49AD8E07-3BF9-A4E2-CB8F-C3722F837B62"
feeder = cim.Feeder(mRID = feeder_mrid, name = 'ieee13')
feeder_model = FeederModel(connection=xml_file, container=feeder, distributed=False)

In [4]:
fdr = cim.Feeder()
fdr.uuid()
json.loads(fdr.__repr__())

{'@id': 'ad2e2882-2b90-4a3f-9754-91d283e5cb06', '@type': 'Feeder'}

In [5]:
from cimtopology.distribution import create_distributed_feeder

create_distributed_feeder(feeder_model, 'ieee13_distributed.xml')

In [6]:
feeder_model.graph[cim.Substation][UUID('6c62c905-6fc7-653d-9f1e-1340f974a587')].pprint()

{
    "@id": "6c62c905-6fc7-653d-9f1e-1340f974a587",
    "@type": "Substation",
    "identifier": "6c62c905-6fc7-653d-9f1e-1340f974a587",
    "mRID": "6C62C905-6FC7-653D-9F1E-1340F974A587",
    "name": "IEEE13",
    "NormalEnergizedFeeder": [
        {
            "@id": "49ad8e07-3bf9-a4e2-cb8f-c3722f837b62",
            "@type": "Feeder"
        }
    ],
    "Region": {
        "@id": "abeb635f-729d-24bf-b8a4-e2ef268d8b9e",
        "@type": "SubGeographicalRegion"
    },
    "SchedulingArea": {
        "@id": "94dfdc1c-8b80-4e90-96e1-504305eddcc2",
        "@type": "DistributionArea"
    }
}


In [7]:
utils.write_xml(feeder_model, 'ieee13_distributed.xml')

In [8]:
from cimgraph.databases import BlazegraphConnection
from cimloader.databases import BlazegraphConnection as BG
from cimloader.uploaders import BlazegraphUploader

params = ConnectionParameters(url = "http://localhost:8889/bigdata/namespace/kb/sparql", cim_profile='cimhub_2023', iec61970_301=8)
bg = BG(params)
loader = BlazegraphUploader(params)
blazegraph = BlazegraphConnection(params)
# bg.drop_all()

# loader.upload_from_file(filepath='./', filename='ieee13_distributed.xml')
# loader.upload_from_file(filepath="/home/ande188/Powergrid-Models/models/feeders/CIM/XML/IEEE13", filename="IEEE13.xml")

# loader.upload_from_file(filepath="/home/ande188/Powergrid-Models/models/feeders/CIM/XML/IEEE13", filename="measurements.xml")

In [9]:
from gridappsd import GridAPPSD
os.environ['GRIDAPPSD_USER'] = 'app_user'
os.environ['GRIDAPPSD_PASSWORD'] = '1234App'
gapps = GridAPPSD()
assert gapps.connected


In [10]:
topic = "goss.gridappsd.request.data.cimtopology"

message = {
   "requestType": "GET_DISTRIBUTED_AREAS",
   "mRID":  "49AD8E07-3BF9-A4E2-CB8F-C3722F837B62",
   "resultFormat": "JSON"
}

topo_message = gapps.get_response(topic, message, timeout=30)

print(json.dumps(topo_message, indent=4))


{
    "DistributionArea": {
        "mRID": "94DFDC1C-8B80-4E90-96E1-504305EDDCC2",
        "Substations": [
            {
                "@id": "6c62c905-6fc7-653d-9f1e-1340f974a587",
                "@type": "Substation"
            }
        ],
        "Feeders": [
            {
                "@id": "49ad8e07-3bf9-a4e2-cb8f-c3722f837b62",
                "@type": "Feeder"
            }
        ],
        "FeederAreas": [
            {
                "mRID": "B6373777-16CB-45D7-9BF0-6F51D91CF949",
                "Feeder": {
                    "@id": "49ad8e07-3bf9-a4e2-cb8f-c3722f837b62",
                    "@type": "Feeder"
                },
                "AddressableEquipment": [
                    {
                        "@id": "52de9189-20dc-4c73-bdee-e960fe1f9493",
                        "@type": "Breaker"
                    },
                    {
                        "@id": "ce5d0651-676b-4af3-8d67-41bf1b33e30c",
                        "@type": "Recloser"
 

In [6]:

dist_area = utils.create_object(cim.DistributionArea, topo_message['DistributionArea']['mRID'] )

dist_model = DistributedArea(container=dist_area, connection=blazegraph, distributed=True)

In [ ]:
for json_ld in topo_message['DistributionArea']['Substations']:
    print('Next Substation')
    substation = dist_model.add_jsonld_to_graph(json_ld)
    SubstationArea = DistributedArea(container=substation, connection=blazegraph, distributed=True)
    SubstationArea.get_all_edges(cim.Substation)
    SubstationArea.get_all_edges(cim.Feeder)
    SubstationArea.get_all_edges(cim.ConnectivityNode)
    SubstationArea.get_all_edges(cim.Terminal)
    SubstationArea.get_all_edges(cim.Analog)
    SubstationArea.get_all_edges(cim.Discrete)

    SubstationArea.pprint(cim.Analog)
    SubstationArea.pprint(cim.Discrete)



    # name = blazegraph.get_from_triple(equipment, "IdentifiedObject.name", {})
    # print(equipment.__class__.__name__, name[0])

In [ ]:
dist_model.graph.keys()

In [ ]:
for json_ld in topo_message['DistributionArea']['FeederAreas'][0]['AddressableEquipment']:
    equipment = dist_model.add_jsonld_to_graph(json_ld)
    name = blazegraph.get_from_triple(equipment, "IdentifiedObject.name", {})
    print(equipment.__class__.__name__, name[0])

In [15]:
with open('ieee13_topo_msg.json','w') as file:
    json.dump(topo_message, file, indent = 4)

In [ ]:
for mrid in topo_message['DistributionArea']['FeederAreas'][0]['AddressableEquipment']:
    equipment = blazegraph.get_object(mrid)
    name = blazegraph.get_from_triple(equipment, "IdentifiedObject.name", {})
    print(equipment.__class__.__name__, name[0])

In [ ]:
for mrid in topo_message['DistributionArea']['FeederAreas'][0]['UnaddressableEquipment']:
    equipment = blazegraph.get_object(mrid)
    name = blazegraph.get_from_triple(equipment, "IdentifiedObject.name", {})
    print(equipment.__class__.__name__, name[0])

In [ ]:
for mrid in topo_message['DistributionArea']['FeederAreas'][0]['Measurements']:
    equipment = blazegraph.get_object(mrid)
    name = blazegraph.get_from_triple(equipment, "IdentifiedObject.name", {})
    print(equipment.__class__.__name__, name[0])

In [3]:
from uuid import UUID
jsonld = dict["@id":UUID,"@type":str(type)]


In [19]:
distribution_area = blazegraph.get_object(mrid='067248c5-53c4-43e8-8720-f0c605c05b84')

In [21]:
distribution_area_model = DistributedArea(container=distribution_area,
                                            connection=blazegraph, distributed=True)

In [ ]:
print(distribution_area_model.get_edges_query(cim.DistributionArea))

In [ ]:
distribution_area_model.get_all_edges(cim.DistributionArea)